# Phase 6.5 sci-Plex 3 registered run

This notebook runs Tasks 6.5.2--6.5.4 on the registered sci-Plex 3 data. It is deliberately conservative about memory: one worker, BLAS threads pinned to one, and a hard preflight check that refuses to start below 5 GiB available RAM. The run emits the seven-arm table, the DTM `k` ablation, held-out bridge diagnostics, and corruption-geometry diagnostics.

Use the repository containing the applied Phase-6.5 modules and the registered `srivatsan_2020_sciplex3.h5ad`. The run is not an invitation to change the estimand, split, grid, or thresholds.

In [ ]:
# Install only the packages needed by the applied runner.
%pip -q install anndata h5py gudhi joblib

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Option A: clone the repository after commit 250ab0c (or a later commit).
# Option B: set REPO_DIR to a Drive copy of this repository if the commit
# has not yet been pushed to GitHub.
REPO_DIR = Path('/content/btate')
if not (REPO_DIR / 'btate/applied/run_phase6_5.py').exists():
    subprocess.run(['git', 'clone', 'https://github.com/hugogobato/btate.git', str(REPO_DIR)], check=True)
assert (REPO_DIR / 'btate/applied/run_phase6_5.py').exists(), (
    'Upload or clone the repository containing btate/applied/run_phase6_5.py'
)

# Mount Drive if the 2.4 GB h5ad is stored there, then edit this path.
H5AD = Path('/content/drive/MyDrive/srivatsan_2020_sciplex3.h5ad')
OUT = Path('/content/phase6_5_sciplex3_results')
assert H5AD.exists(), f'Missing registered data file: {H5AD}'
print('repo:', REPO_DIR)
print('data:', H5AD, H5AD.stat().st_size / 2**30, 'GiB')


In [ ]:
# Conservative resource guard. Do not raise JOBS on a shared runtime.
import psutil
available = psutil.virtual_memory().available
minimum = 5 * 2**30
if available < minimum:
    raise RuntimeError(f'Only {available / 2**30:.2f} GiB available; refusing to risk OOM')

os.environ['OMP_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['NUMEXPR_NUM_THREADS'] = '1'

JOBS = 2 if available >= 10 * 2**30 else 1
FRAME_CAP = 50_000
N_REPLICATES = 100
N_REP = 20
print(f'available RAM: {available / 2**30:.2f} GiB; jobs={JOBS}')


In [ ]:
# Registered full run. This may take several hours because every one of the
# 4,614 calibration units receives all retention-level Alpha/DTM curves.
cmd = [
    sys.executable, '-u', '-m', 'btate.applied.run_phase6_5',
    '--h5', str(H5AD), '--out', str(OUT), '--jobs', str(JOBS),
    '--frame-cap', str(FRAME_CAP), '--n-replicates', str(N_REPLICATES),
    '--n-rep', str(N_REP),
]
subprocess.run(cmd, cwd=REPO_DIR, check=True)


In [ ]:
# Mechanical completeness checks before reading any scientific conclusion.
import json
import pandas as pd

artifacts = json.loads((OUT / 'artifacts.json').read_text())
assert artifacts['n_eval'] == 27
assert artifacts['n_calibration'] == 4614
assert artifacts['n_excluded'] == 3
main = pd.read_csv(OUT / 'replicate_rows.csv')
ablation = pd.read_csv(OUT / 'dtm_ablation_rows.csv')
assert len(main) == 100 * 4 * 7
assert len(ablation) == 100 * 8
assert not bool(main['failed'].any())
assert not bool(ablation['failed'].any())
print(pd.read_csv(OUT / 'aggregate.csv').to_string(index=False))
print('DTM ablation:')
print(pd.read_csv(OUT / 'dtm_ablation_aggregate.csv').to_string(index=False))


In [ ]:
# Bundle all artifacts and download automatically when running in Colab.
import shutil
archive = shutil.make_archive('/content/phase6_5_sciplex3_artifacts', 'zip', root_dir=OUT)
output_file = archive
try:
    from google.colab import files
    files.download(output_file)
    print('Downloaded:', output_file)
except Exception as e:
    print('(Not on Colab / download skipped):', e)
print('Artifacts:', OUT)
